<a href="https://colab.research.google.com/github/joelmanrique91-lgtm/geostats-colab-lab/blob/main/PCA_notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Uploading CSV Data

First, we'll use `google.colab.files.upload()` to let you select and upload your CSV file. After running the next cell, a 'Choose Files' button will appear. Click it and select your CSV file.

In [ ]:
import pandas as pd
import datetime

# Initialize a global list to store audit messages
audit_log = []

def log_audit_message(message, dataframe_name=None, dataframe_instance=None, transformation_type=None, details=None):
    """
    Logs an audit message, optionally including DataFrame state and additional details.

    Args:
        message (str): A descriptive message for the audit entry.
        dataframe_name (str, optional): The name of the DataFrame being audited.
        dataframe_instance (pd.DataFrame, optional): The DataFrame object itself.
        transformation_type (str, optional): The type of transformation applied (e.g., 'Load', 'Drop Columns', 'CLR').
        details (dict, optional): A dictionary for additional, custom details to log.
    """
    audit_entry = {
        'timestamp': pd.Timestamp.now(),
        'message': message,
        'transformation_type': transformation_type
    }
    if dataframe_instance is not None:
        audit_entry['dataframe_name'] = dataframe_name
        audit_entry['shape'] = dataframe_instance.shape
        audit_entry['columns'] = dataframe_instance.columns.tolist()
        audit_entry['dtypes'] = {col: str(dtype) for col, dtype in dataframe_instance.dtypes.items()}

    if details is not None:
        audit_entry['details'] = details

    audit_log.append(audit_entry)
    print(f"Audit Log: {message}")

In [ ]:
import pandas as pd
from google.colab import files
import io

print('Please upload your CSV file with XYZ coordinates:')

uploaded = files.upload()

for fn in uploaded.keys():
  print(f'User uploaded file "{fn}" with length {len(uploaded[fn])} bytes')
  # Read the CSV into a pandas DataFrame
  df = pd.read_csv(io.StringIO(uploaded[fn].decode('utf-8')))
  print(f'Successfully loaded "{fn}" into a DataFrame.')


# Display the first 5 rows and info to confirm it loaded correctly
print('\nFirst 5 rows of the DataFrame:')
display(df.head())

print('\nDataFrame Info:')
df.info()

### Selección Interactiva de Columnas

Aquí puedes seleccionar las columnas que no te interesan para eliminarlas del DataFrame `df`. Utiliza la lista desplegable para seleccionar una o varias columnas.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

def drop_selected_columns(df_current):
    all_columns = df_current.columns.tolist()

    # Create a multi-select widget
    column_selector = widgets.SelectMultiple(
        options=all_columns,
        value=[],
        description='Columnas a eliminar:',
        disabled=False
    )

    # Button to confirm selection
    confirm_button = widgets.Button(
        description='Eliminar Columnas',
        disabled=False,
        button_style='success',
        tooltip='Click para eliminar las columnas seleccionadas'
    )

    output = widgets.Output()

    def on_button_click(b):
        with output:
            clear_output()
            columns_to_drop = list(column_selector.value)
            if columns_to_drop:
                print(f"Eliminando las siguientes columnas: {columns_to_drop}")
                global df # Access the global df
                df = df_current.drop(columns=columns_to_drop, errors='ignore')
                print("Columnas eliminadas con éxito.")
                print("\nPrimeras 5 filas del DataFrame actualizado:")
                display(df.head())
                print("\nInformación del DataFrame actualizado:")
                df.info()
            else:
                print("No se seleccionaron columnas para eliminar.")

    confirm_button.on_click(on_button_click)

    display(column_selector, confirm_button, output)

drop_selected_columns(df.copy()) # Pass a copy to avoid modifying df directly before user confirmation

## 1. Preparación de Datos

Continuando con el plan, ahora realizaremos la preparación de los datos mineralógicos para el análisis de PCA.

### 1.1 Identificación de Columnas Mineralógicas

Primero, necesitamos identificar y seleccionar solo las columnas que representan abundancias mineralógicas. Basado en tu contexto, estas son las columnas que se utilizarán para el PCA.

In [ ]:
import pandas as pd
import numpy as np # Ensure numpy is imported for nan-related checks, if not already

# Helper function to check for problematic columns
def check_problematic_columns(df_check, mineral_cols_to_check):
    problematic_cols = []
    for col in mineral_cols_to_check:
        if col not in df_check.columns:
            continue

        # Check for completely empty columns
        if df_check[col].isnull().all():
            problematic_cols.append((col, 'Completely empty (all NaN)'))
            print(f"WARNING: Column '{col}' is completely empty and will be excluded.")
        # Check for constant columns
        elif df_check[col].nunique() == 1:
            problematic_cols.append((col, 'Constant (all values are the same)'))
            print(f"WARNING: Column '{col}' is constant and will be excluded.")
        # Check for high NaN percentage
        elif df_check[col].isnull().sum() / len(df_check) > 0.5:
            problematic_cols.append((col, 'More than 50% NaN'))
            print(f"WARNING: Column '{col}' has more than 50% NaN values and will be excluded.")
    return problematic_cols

print("--- Dynamic Identification of Mineralogical Columns ---")
log_audit_message("Starting dynamic mineral column identification.", transformation_type="Dynamic Column Identification")

# 1. Log all initial columns of DataFrame 'df'
all_initial_df_columns = df.columns.tolist()
print(f"Initial columns in 'df': {all_initial_df_columns}")
log_audit_message(
    "Recorded initial DataFrame columns.",
    dataframe_name="df",
    dataframe_instance=df,
    transformation_type="Initial Column Scan"
)

# 2. Dynamically detect mineralogical columns (e.g., with prefix 'MINCALC_')
# Criteria: The column must exist in df and start with 'MINCALC_'
potential_mineral_cols = [col for col in all_initial_df_columns if col.startswith("MINCALC_")]

print(f"\nCandidate mineralogical columns (prefix 'MINCALC_'): {potential_mineral_cols}")
log_audit_message(
    f"Identified {len(potential_mineral_cols)} candidate mineral columns based on 'MINCALC_' prefix.",
    details={"candidate_columns": potential_mineral_cols},
    transformation_type="Candidate Column Detection"
)

# Identify columns excluded for not following naming convention
excluded_by_naming_convention = [col for col in all_initial_df_columns if not col.startswith("MINCALC_")]
if excluded_by_naming_convention:
    print(f"Columns excluded for not following 'MINCALC_' naming convention: {excluded_by_naming_convention}")
    log_audit_message(
        f"Excluded {len(excluded_by_naming_convention)} columns from mineralogical analysis for not following 'MINCALC_' naming convention.",
        details={"excluded_columns": excluded_by_naming_convention},
        dataframe_name="df",
        dataframe_instance=df,
        transformation_type="Naming Convention Exclusion"
    )

# 3. Identify and exclude problematic columns among candidates
problematic_cols_details = check_problematic_columns(df, potential_mineral_cols)

mineral_cols = potential_mineral_cols.copy() # Start with all potential mineral columns
if problematic_cols_details:
    excluded_by_problem = [col for col, _ in problematic_cols_details]
    mineral_cols = [col for col in potential_mineral_cols if col not in excluded_by_problem]
    print(f"Columns excluded due to being problematic: {excluded_by_problem}")
    log_audit_message(
        f"Excluded {len(excluded_by_problem)} columns due to problematic data (empty, constant, or high NaN percentage).",
        details={"excluded_columns": problematic_cols_details},
        dataframe_name="df",
        dataframe_instance=df,
        transformation_type="Problematic Column Exclusion"
    )
else:
    print("No problematic columns identified among candidates.")
    log_audit_message(
        "No problematic columns identified among candidate mineralogical columns.",
        dataframe_name="df",
        dataframe_instance=df,
        transformation_type="Problematic Column Check"
    )

print(f"\nFinal list of mineralogical columns for PCA: {mineral_cols}")
log_audit_message(
    f"Final list of {len(mineral_cols)} mineralogical columns selected for PCA.",
    details={"final_mineral_columns": mineral_cols},
    dataframe_name="df",
    dataframe_instance=df,
    transformation_type="Final Mineral Column Selection"
)

# Create df_minerals DataFrame
df_minerals = df[mineral_cols].copy()
log_audit_message(
    "Created initial df_minerals DataFrame from selected columns.",
    dataframe_name="df_minerals",
    dataframe_instance=df_minerals,
    transformation_type="df_minerals Creation"
)

# Handle NaNs: drop rows with any NaN values
initial_rows_df_minerals = df_minerals.shape[0]
df_minerals_cleaned = df_minerals.dropna().copy()
dropped_rows_count = initial_rows_df_minerals - df_minerals_cleaned.shape[0]

if dropped_rows_count > 0:
    print(f"Dropped {dropped_rows_count} rows from df_minerals due to NaN values.")
    log_audit_message(
        f"Dropped {dropped_rows_count} rows from df_minerals due to NaN values.",
        dataframe_name="df_minerals",
        dataframe_instance=df_minerals_cleaned,
        transformation_type="NaN Row Dropping"
    )
else:
    print("No rows dropped from df_minerals due to NaN values.")
    log_audit_message(
        "No rows dropped from df_minerals due to NaN values.",
        dataframe_name="df_minerals",
        dataframe_instance=df_minerals_cleaned,
        transformation_type="NaN Row Dropping (No Change)"
    )

df_minerals = df_minerals_cleaned # Update df_minerals to the cleaned version

print(f"\nFinal df_minerals shape after NaN handling: {df_minerals.shape}")
print("First 5 rows of df_minerals:")
display(df_minerals.head())
print("\nInfo of df_minerals:")
df_minerals.info()

# Explicitly log how 'MINCALC_TOT_FELDSPAR' was handled in this dynamic process.
# We need to check both if it was in initial df columns and if it ended up in mineral_cols.
mincalc_tot_feldspar_initial_presence = 'MINCALC_TOT_FELDSPAR' in all_initial_df_columns
mincalc_tot_feldspar_final_presence = 'MINCALC_TOT_FELDSPAR' in mineral_cols

if mincalc_tot_feldspar_initial_presence:
    if not mincalc_tot_feldspar_final_presence:
        if any('MINCALC_TOT_FELDSPAR' == col for col, _ in problematic_cols_details):
            handling_message = "'MINCALC_TOT_FELDSPAR' was initially present but excluded because it was identified as problematic (e.g., constant, empty, or high NaN percentage)."
        elif not 'MINCALC_TOT_FELDSPAR'.startswith("MINCALC_"): # This case is unlikely given the variable name
             handling_message = "'MINCALC_TOT_FELDSPAR' was initially present but excluded because it did not follow the 'MINCALC_' naming convention (this should not happen for this column name)."
        else:
            handling_message = "'MINCALC_TOT_FELDSPAR' was initially present but excluded for an unspecified reason during dynamic column selection."
    else:
        handling_message = "'MINCALC_TOT_FELDSPAR' was initially present and is included in the final mineralogical columns."
else:
    handling_message = "'MINCALC_TOT_FELDSPAR' was not present in the initial DataFrame columns, so it was not considered for PCA."

print(f"\nHandling of 'MINCALC_TOT_FELDSPAR': {handling_message}")
log_audit_message(
    f"Handling of 'MINCALC_TOT_FELDSPAR': {handling_message}",
    transformation_type="MINCALC_TOT_FELDSPAR Handling Report"
)

### 1.2 Manejo de Ceros (Pseudocount)

La transformación log-ratio no admite valores cero. Aplicaremos un pequeño pseudocount (`epsilon = 1e-6`) a todas las abundancias minerales para evitar errores matemáticos y mantener la proporcionalidad.

In [ ]:
epsilon = 1e-6
print(f"Aplicando pseudocount de {epsilon} a las columnas mineralógicas con valor 0...")
df_minerals = df_minerals.replace(0, epsilon)

# Verificar si todavía hay ceros (debería ser cero)
num_zeros_after_replace = (df_minerals == 0).sum().sum()
print(f"Número de valores cero después de aplicar pseudocount: {num_zeros_after_replace}")

### 1.3 Normalización Composicional (Cierre Composicional)

Para tratar el dataset como composicional, aplicamos un cierre composicional. Esto asegura que la suma de las abundancias de minerales en cada fila sea aproximadamente 1 (o 100%, dependiendo de la escala). Esto es crucial para un análisis PCA válido en datos composicionales.

In [ ]:
print("Aplicando cierre composicional...")
row_sum = df_minerals[mineral_cols].sum(axis=1)
df_closed = df_minerals[mineral_cols].div(row_sum, axis=0)

# Verificar que cada fila suma aproximadamente 1.0
print("Verificando la suma de las filas después del cierre composicional (primeras 5 filas):")
display(df_closed.head().sum(axis=1))

# También puedes verificar la suma de todas las filas
# print(df_closed.sum(axis=1).describe())

### 1.4 Transformación CLR (Centered Log Ratio)

Finalmente, aplicamos la transformación Centered Log Ratio (CLR). Esta transformación es fundamental para el análisis de datos composicionales, ya que convierte las proporciones en un espacio euclidiano, haciendo que sean adecuadas para técnicas estadísticas multivariadas como PCA. El dataset resultante `X_clr` será el que utilizaremos para el PCA.

In [ ]:
from scipy.stats import gmean
import numpy as np

print("Aplicando transformación Centered Log Ratio (CLR)...")

# Calcular la media geométrica para cada fila
geometric_mean = gmean(df_closed, axis=1)

# Aplicar la transformación CLR
X_clr = np.log(df_closed.div(geometric_mean, axis=0))

# Convertir X_clr a un DataFrame de pandas para facilitar el seguimiento
X_clr = pd.DataFrame(X_clr, columns=mineral_cols, index=df_closed.index)

print("Primeras 5 filas del DataFrame CLR transformado (X_clr):")
display(X_clr.head())

print("Información del DataFrame CLR transformado (X_clr):")
X_clr.info()

## 4. Estadísticas Obligatorias

Analizaremos los resultados del PCA para entender la contribución de cada componente y la relación con las variables mineralógicas.

### 4.1 Tabla de Varianza Explicada

Esta tabla nos muestra la varianza explicada por cada componente principal, tanto en porcentaje individual como acumulado. Es fundamental para determinar cuántos componentes contienen la señal útil de nuestros datos.

In [ ]:
import numpy as np

# Calcular eigenvalues a partir de singular_values_
eigenvalues = pca.singular_values_**2 / (X_scaled.shape[0] - 1)

# Crear la tabla de varianza explicada
explained_variance_df = pd.DataFrame({
    'PC': [f'PC{i+1}' for i in range(len(explained_variance_ratio))],
    'Eigenvalue': eigenvalues,
    'Varianza %': explained_variance_ratio * 100,
    'Acumulada %': np.cumsum(explained_variance_ratio) * 100
})

print("Tabla de Varianza Explicada:")
display(explained_variance_df)

# Objetivo geológico: evaluar cuántas PCs contienen señal útil
print("\nPara una interpretación geológica, es común buscar las primeras 2-5 PCs que explican una parte significativa de la varianza.")

### 4.2 Scree Plot

El Scree Plot visualiza los eigenvalues de cada componente principal. Nos ayuda a identificar el 'codo' (elbow) en el gráfico, que sugiere un punto donde la varianza explicada por componentes adicionales disminuye drásticamente, separando la 'señal' del 'ruido'. También añadiremos la línea del Criterio de Kaiser (eigenvalue = 1).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 6))
sns.lineplot(x=range(1, len(eigenvalues) + 1), y=eigenvalues, marker='o')
plt.axhline(y=1, color='r', linestyle='--', label='Criterio de Kaiser (Eigenvalue = 1)')
plt.title('Scree Plot - Varianza Explicada por Componente Principal')
plt.xlabel('Número de Componente Principal')
plt.ylabel('Eigenvalue (Varianza)')
plt.xticks(range(1, len(eigenvalues) + 1))
plt.grid(True, linestyle='--', alpha=0.6)
plt.legend()
plt.show()

print("\nInterpretación del Scree Plot: El 'codo' en la gráfica y el Criterio de Kaiser (eigenvalues > 1) nos ayudan a seleccionar el número óptimo de componentes principales a retener.")

### 4.3 Cargas (Loadings)

Las cargas (loadings) representan la correlación entre cada variable original (mineral) y los componentes principales. Son fundamentales para la interpretación geológica, ya que nos indican qué minerales contribuyen más a cada componente y en qué dirección. Valores positivos y negativos indican asociaciones u oposiciones.

In [ ]:
print("Tabla de Cargas (Loadings) por Componente Principal:")
# pca_components_df ya fue creado en el paso 3 del PCA
display(pca_components_df)

print("\nInterpretación: Observa los minerales con las cargas más altas (positivas o negativas) en cada componente. Minerales con cargas positivas altas en el mismo PC tienden a estar asociados. Minerales con cargas opuestas sugieren un gradiente mineralógico.")

### 4.4 Contribution Plot

El Contribution Plot muestra la contribución relativa de cada mineral a la varianza explicada por cada componente principal. Es muy útil para identificar los minerales que más influyen en cada PC, ordenados por magnitud absoluta de su contribución. Esto es fundamental para la interpretación hidrotermal.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

def plot_contributions(components_df, explained_variance_ratio, num_pcs=3, top_n_minerals=10):
    for i in range(num_pcs):
        pc_label = f'PC{i+1}'
        # Calculamos la contribución de cada mineral como el cuadrado de la carga dividido por el eigenvalue correspondiente
        # Otra forma común de 'contribución' es el valor absoluto de la carga
        # Para este caso, siguiendo la descripción, nos enfocaremos en la magnitud de la carga.

        # Crear un DataFrame para las cargas de la PC actual
        loadings_pc = pd.DataFrame({
            'Mineral': components_df.columns,
            'Loading': components_df.loc[pc_label].values
        })

        # Ordenar por magnitud absoluta de la carga
        loadings_pc['Abs_Loading'] = loadings_pc['Loading'].abs()
        loadings_pc = loadings_pc.sort_values(by='Abs_Loading', ascending=False).head(top_n_minerals)

        plt.figure(figsize=(12, 6))
        sns.barplot(x='Mineral', y='Loading', data=loadings_pc, palette='coolwarm')
        plt.title(f'Contribución de Minerales a {pc_label} (Top {top_n_minerals})\nVarianza Explicada: {explained_variance_ratio[i]*100:.2f}%')
        plt.xlabel('Mineral')
        plt.ylabel('Carga (Loading)')
        plt.xticks(rotation=45, ha='right')
        plt.grid(axis='y', linestyle='--', alpha=0.7)
        plt.tight_layout()
        plt.show()

# Puedes ajustar el número de PCs a mostrar y la cantidad de minerales principales
plot_contributions(pca_components_df, explained_variance_ratio, num_pcs=5, top_n_minerals=10)

print("\nInterpretación: Estos gráficos de barras muestran los minerales que tienen la mayor influencia (cargas más altas en magnitud) en cada uno de los primeros componentes principales. Esto ayuda a visualizar las asociaciones o antagonismos mineralógicos clave de cada PC.")

## 5. Gráficos Principales

Pasamos a la visualización de los resultados del PCA, que nos permitirán una interpretación más intuitiva de las asociaciones mineralógicas y los gradientes hidrotermales.

### 5.1 Score Plot PC1 vs PC2

Este gráfico de dispersión muestra la posición de cada muestra en el espacio de los dos primeros componentes principales (PC1 y PC2). Es útil para identificar clusters naturales, gradientes hidrotermales y mezclas litológicas. Podemos opcionalmente colorear los puntos por alguna variable categórica como la litología, alteración o profundidad si están disponibles en el DataFrame original `df`.

Para poder usar colores basados en variables como 'litología' o 'alteración', primero necesito saber si tienes esas columnas en tu DataFrame `df`. Si las tienes, podemos utilizarlas para colorear el gráfico y hacer una interpretación más rica. Por ahora, generaré un gráfico básico, y luego podemos mejorarlo si me indicas las columnas de interés.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10, 8))
sns.scatterplot(x='PC1', y='PC2', data=pca_scores_df, s=20, alpha=0.7)
plt.title('Score Plot: PC1 vs PC2')
plt.xlabel(f'PC1 ({explained_variance_ratio[0]*100:.2f}% de varianza explicada)')
plt.ylabel(f'PC2 ({explained_variance_ratio[1]*100:.2f}% de varianza explicada)')
plt.grid(True, linestyle='--', alpha=0.6)
plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
plt.show()

print("\nInterpretación: Este gráfico muestra la distribución de las muestras en el espacio de los dos primeros componentes principales. Los puntos cercanos entre sí indican muestras con composiciones mineralógicas similares.")

# Opcional: Si tienes columnas como 'litología' o 'alteración' en el DataFrame 'df',
# podemos usarlas para colorear los puntos y mejorar la interpretación. Por ejemplo:
# sns.scatterplot(x='PC1', y='PC2', hue=df['litologia_columna'], data=pca_scores_df, s=20, alpha=0.7)
# Si deseas, dime qué columna de 'df' te gustaría usar para colorear el gráfico.

### 5.2 Biplot

El biplot es una herramienta poderosa que muestra tanto las muestras (puntos) como las variables originales (minerales, como vectores) en el espacio de los componentes principales. Esto permite visualizar directamente las relaciones entre las muestras y la influencia de cada mineral en los componentes principales.

*   **Vectores minerales:** La dirección de un vector indica cómo un mineral contribuye al componente principal. Los minerales que apuntan en la misma dirección están asociados. La longitud del vector indica la fuerza de esa contribución.
*   **Muestras:** La posición de una muestra indica su composición mineralógica. Las muestras cerca de un vector mineral tienen una mayor abundancia relativa de ese mineral.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

def biplot(score, coeff, labels=None, score_labels=None):
    xs = score[:,0] # PC1 scores
    ys = score[:,1] # PC2 scores
    n = coeff.shape[0] # Number of features
    scalex = 1.0/(xs.max() - xs.min()) # Scale for scores
    scaley = 1.0/(ys.max() - ys.min())

    fig = plt.figure(figsize=(12, 12))
    ax = fig.add_subplot(111)

    # Plot scores
    ax.scatter(xs * scalex, ys * scaley, s=20, alpha=0.7)

    # Plot loadings (arrows)
    for i in range(n):
        ax.arrow(0, 0, coeff[i,0], coeff[i,1], color='r', alpha=0.8, head_width=0.03, head_length=0.03)
        if labels is None:
            ax.text(coeff[i,0]*1.15, coeff[i,1]*1.15, f"Var{i+1}", color='g', ha='center', va='center')
        else:
            ax.text(coeff[i,0]*1.15, coeff[i,1]*1.15, labels[i], color='r', ha='center', va='center', fontsize=9)

    ax.set_xlabel(f'PC1 ({explained_variance_ratio[0]*100:.2f}% de varianza explicada)')
    ax.set_ylabel(f'PC2 ({explained_variance_ratio[1]*100:.2f}% de varianza explicada)')
    ax.set_title('Biplot: PC1 vs PC2 con Vectores Minerales')
    ax.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    plt.tight_layout()
    plt.show()

# Obtener los scores de PCA y los componentes (loadings) para PC1 y PC2
score_matrix = pca_scores_df.values[:, :2] # Scores para PC1 y PC2

# Asegurarse de que coeff_matrix tenga las dimensiones correctas
# Los loadings para PC1 y PC2 son las primeras dos filas de pca_components_df
# y queremos las características (minerales) como filas y las PCs como columnas.
coeff_matrix = pca_components_df.loc[['PC1', 'PC2'], :].T.values # Loadings para PC1 y PC2

# Crear el biplot
biplot(score_matrix, coeff_matrix, labels=mineral_cols)

print("\nInterpretación: El biplot muestra cómo las muestras se agrupan en relación con las influencias de los distintos minerales. Observa la dirección y longitud de los vectores minerales para entender sus asociaciones y su importancia en cada componente principal.")

### 5.3 Correlation Circle

El Círculo de Correlación (Correlation Circle) es una visualización que muestra las variables originales (minerales) proyectadas en el espacio de los componentes principales, con sus vectores normalizados. Es particularmente útil para:

*   **Asociaciones minerales:** Los minerales cuyos vectores están cerca entre sí (ángulos pequeños) están positivamente correlacionados.
*   **Antagonismos:** Los minerales con vectores en direcciones opuestas (ángulos cercanos a 180 grados) están negativamente correlacionados.
*   **Importancia:** La longitud del vector hasta el borde del círculo indica qué tan bien representada está la variable por los componentes principales mostrados.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

def plot_correlation_circle(pca_components_df, explained_variance_ratio, feature_names, pc_x=1, pc_y=2):
    # Obtener los loadings para PC_x y PC_y
    pc_x_idx = pc_x - 1
    pc_y_idx = pc_y - 1

    if pc_x_idx >= len(pca_components_df) or pc_y_idx >= len(pca_components_df):
        print(f"Error: Los componentes PC{pc_x} o PC{pc_y} no existen.")
        return

    loadings_x = pca_components_df.loc[f'PC{pc_x}'].values
    loadings_y = pca_components_df.loc[f'PC{pc_y}'].values

    plt.figure(figsize=(10, 10))

    # Dibujar el círculo de correlación
    circle = plt.Circle((0, 0), 1, color='gray', fill=False, linestyle='--')
    plt.gca().add_artist(circle)

    # Dibujar los vectores para cada mineral
    for i, feature in enumerate(feature_names):
        x = loadings_x[i]  # Loading para PCx
        y = loadings_y[i]  # Loading para PCy
        plt.arrow(0, 0, x, y, color='b', alpha=0.8, head_width=0.03, head_length=0.03)
        plt.text(x * 1.15, y * 1.15, feature, color='g', ha='center', va='center', fontsize=9)

    plt.xlim(-1.1, 1.1)
    plt.ylim(-1.1, 1.1)
    plt.xlabel(f'PC{pc_x} ({explained_variance_ratio[pc_x_idx]*100:.2f}% de varianza explicada)')
    plt.ylabel(f'PC{pc_y} ({explained_variance_ratio[pc_y_idx]*100:.2f}% de varianza explicada)')
    plt.title(f'Círculo de Correlación: PC{pc_x} vs PC{pc_y}')
    plt.grid(True, linestyle='--', alpha=0.6)
    plt.axhline(0, color='grey', linestyle='--', linewidth=0.8)
    plt.axvline(0, color='grey', linestyle='--', linewidth=0.8)
    plt.tight_layout()
    plt.show()

# Llamar a la función para el círculo de correlación
plot_correlation_circle(pca_components_df, explained_variance_ratio, mineral_cols, pc_x=1, pc_y=2)

print("\nInterpretación: Los minerales con vectores cercanos indican correlación positiva. Los vectores opuestos indican correlación negativa. La longitud del vector indica cuán bien la variable es representada por los componentes principales.")

### 5.4 Heatmap de Loadings

Un heatmap de las cargas es una excelente forma de visualizar rápidamente qué minerales contribuyen de manera significativa a cada componente principal. Los colores representarán la magnitud y el signo de la carga, permitiendo identificar patrones de asociación y oposición.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(12, 8))
sns.heatmap(
    pca_components_df.head(len(mineral_cols)), # Mostrar todos los componentes principales que tenemos (len(mineral_cols) es el número total de componentes)
    cmap='coolwarm',
    annot=True,
    fmt=".2f",
    linewidths=.5,
    cbar_kws={'label': 'Carga (Loading)'}
)
plt.title('Heatmap de Loadings de Componentes Principales')
plt.xlabel('Minerales')
plt.ylabel('Componente Principal')
plt.xticks(rotation=45, ha='right')
plt.yticks(rotation=0)
plt.tight_layout()
plt.show()

print("\nInterpretación: En este heatmap, los colores cálidos (rojos) indican cargas positivas altas, y los colores fríos (azules) indican cargas negativas altas. Los valores cercanos a cero (blanco) indican poca contribución. Esto facilita la identificación de los minerales clave para cada PC.")

## 6. Interpretación Geológica

Es fundamental recordar que las PCs no deben interpretarse automáticamente. Cada componente principal debe analizarse cuidadosamente en el contexto geológico del depósito, considerando:

*   **Litología:** ¿Cómo influye la roca huésped en la composición mineralógica?
*   **Paragénesis:** ¿Qué asociaciones de minerales se esperan en los diferentes eventos hidrotermales?
*   **Alteración:** ¿Qué ensamblajes de alteración son característicos y cómo se manifiestan en las PCs?
*   **Controles geológicos:** ¿Existen estructuras o contactos litológicos que controlen la distribución de la alteración?
*   **Profundidad:** ¿Hay variaciones mineralógicas relacionadas con la profundidad?
*   **Tendencias espaciales:** ¿Se observan patrones en los score plots al mapear los scores en el espacio?

**Posibles Interpretaciones Típicas:**
*   **PC1:** A menudo, representa un contraste entre asociaciones mineralógicas dominantes, como potásica (biotita, feldespato K, magnetita) vs. fílica (muscovita, cuarzo, pirita) o argílica (caolinita, esmectita).
*   **PC2:** Podría reflejar silicificación (cuarzo) o un contraste con minerales máficos (anfíbol, clorita).
*   **PC3:** Podría relacionarse con propilitización (clorita, epidota, calcita) u otras fases de alteración.

**Validar SIEMPRE contra la geología real, la petrografía y la mineralogía conocida del yacimiento.**

## 7. Validaciones Importantes

Para asegurar la robustez de la interpretación, es crucial realizar algunas validaciones adicionales.

### 7.1 Verificar dominancia litológica

Si el Score Plot se colorea por litología y los clusters separan predominantemente litologías, esto podría indicar que el PCA está capturando principalmente la variabilidad de la roca huésped en lugar de la alteración hidrotermal. En estos casos, podría ser más apropiado correr el PCA por litología separada para aislar la señal de alteración.

### 7.2 Verificar minerales dominantes

Revisar si los primeros componentes principales están siendo monopolizados por minerales muy abundantes (como cuarzo o feldespatos). Si su varianza es tan alta que enmascara las señales de otros minerales de alteración más sutiles, puede ser necesario reevaluar la inclusión de estos minerales o considerar una normalización diferente si es aplicable y geológicamente justificado.

### 7.3 Verificar estabilidad

La estabilidad del PCA se puede evaluar repitiendo el análisis con diferentes subconjuntos de datos:
*   Removiendo minerales raros o con baja varianza.
*   Removiendo outliers identificados.
*   Realizando el PCA por dominios geológicos específicos (ej. diferentes cuerpos de mineral, zonas de alteración conocidas).

Comparar las cargas resultantes para ver si las asociaciones minerales se mantienen consistentes. La estabilidad indica una señal robusta.

## 8. Outliers

Los outliers pueden distorsionar significativamente los resultados del PCA. Es importante detectarlos y entender su naturaleza. Podemos identificarlos mediante métricas como:

*   **Leverage:** Indica la influencia de una observación en la regresión (en el contexto de PCA, en la determinación de las direcciones de los componentes).
*   **Distancia de Hotelling T²:** Mide la distancia multivariada de una observación al centroide de los datos, útil para identificar outliers extremos.
*   **Distancia de score:** Mide cuán lejos está una observación de la media de los scores de un componente principal.

La visualización de un `outlier map` puede ayudar a entender su distribución. La decisión de removerlos o no debe ser cuidadosamente considerada y justificada geológicamente.

## 9. Clustering Opcional

Una vez que se han obtenido los scores de PCA y se han interpretado los componentes, se puede realizar un clustering sobre estos scores (y **no** directamente sobre el espacio original) para identificar ensamblajes hidrotermales o dominios de alteración.

Algoritmos comunes incluyen:
*   **KMeans:** Requiere definir el número de clusters de antemano.
*   **HDBSCAN:** Más flexible, no requiere un número predefinido de clusters y puede manejar clusters de formas irregulares.
*   **Clustering jerárquico:** Útil para explorar la estructura de los datos a diferentes niveles de granularidad.

El clustering sobre los scores PCA ayuda a trabajar en un espacio de menor dimensionalidad y con mayor señal/ruido.

## 10. Resultado Final Esperado

El objetivo final de este flujo de PCA es que actúe como una herramienta exploratoria multivariada que permita:

*   **Detectar asociaciones minerales** coherentes con la geología.
*   **Identificar gradientes hidrotermales** y zonas de transición.
*   **Separar ensamblajes** de alteración y tipos de mineralización.
*   **Reconocer destrucción o reemplazo mineral** a través de patrones en los loadings.
*   **Visualizar transiciones alteracionales** en los score plots.
*   **Reducir la dimensionalidad** del conjunto de datos mineralógico, facilitando la interpretación.
*   **Construir dominios hidrotermales preliminares** que luego pueden ser validados con más datos geológicos y espaciales.

**IMPORTANTE:** El PCA no debe usarse como una clasificación final automática, sino como una herramienta integrada con el conocimiento geológico y las tendencias espaciales para construir modelos interpretativos más robustos.

## 2. Escalado

Antes de aplicar PCA, es **muy importante** escalar los datos para que cada característica (mineral) contribuya igualmente al análisis, evitando que minerales con mayor varianza numérica dominen los componentes principales. Esto se logra centrando la media y escalando la varianza a la unidad.

In [ ]:
from sklearn.preprocessing import StandardScaler

print("Aplicando StandardScaler a los datos CLR transformados (X_clr)...")

# Inicializar el StandardScaler
scaler = StandardScaler()

# Ajustar el scaler a los datos y transformarlos
X_scaled = scaler.fit_transform(X_clr)

# Convertir X_scaled de nuevo a un DataFrame para mantener los nombres de las columnas y la indexación
X_scaled = pd.DataFrame(X_scaled, columns=X_clr.columns, index=X_clr.index)

print("Datos escalados con éxito. Primeras 5 filas de X_scaled:")
display(X_scaled.head())

print("Información del DataFrame escalado (X_scaled):")
X_scaled.info()

print("Verificando la media (debería ser cercana a 0) y la desviación estándar (debería ser cercana a 1) para las primeras columnas:")
display(X_scaled.iloc[:, :5].describe())

## 3. PCA (Análisis de Componentes Principales)

Ahora aplicaremos el Análisis de Componentes Principales (PCA) a los datos escalados (`X_scaled`). Inicialmente, utilizaremos `n_components=None` para capturar todos los componentes y poder evaluar la varianza explicada por cada uno.

In [ ]:
from sklearn.decomposition import PCA

print("Realizando el análisis PCA...")

# Inicializar PCA con n_components=None para obtener todos los componentes
pca = PCA(n_components=None)

# Ajustar PCA a los datos escalados
X_pca = pca.fit_transform(X_scaled)

# Guardar los resultados importantes del PCA
explained_variance_ratio = pca.explained_variance_ratio_
components = pca.components_
singular_values = pca.singular_values_

print("PCA completado. Se han calculado los siguientes atributos:")
print(f"  - explained_variance_ratio_ (Varianza explicada por cada componente): {explained_variance_ratio.shape[0]} componentes")
print(f"  - components_ (Componentes principales o 'loadings'): {components.shape[0]} componentes, {components.shape[1]} características")
print(f"  - singular_values_ (Valores singulares): {singular_values.shape[0]} valores")

# Convertir los componentes a un DataFrame para facilitar la visualización
pca_components_df = pd.DataFrame(
    components,
    columns=mineral_cols,
    index=[f'PC{i+1}' for i in range(components.shape[0])]
)

print("\nPrimeras filas de los componentes principales (loadings):")
display(pca_components_df.head())

# Convertir los scores PCA a un DataFrame
pca_scores_df = pd.DataFrame(
    X_pca,
    columns=[f'PC{i+1}' for i in range(X_pca.shape[1])],
    index=X_scaled.index
)